# 국민은행_토스앱 리뷰 1년치 수집_저장_텍스트분석_mini_project
* 1. google play에서 국민은행 스타뱅킹앱, 토스앱의 사용자 리뷰를 1년치 수집
* 2. 수집한 리뷰를 mysql에 저장
* 3. 수집된 데이터로 긍정/부정 리뷰 비율 분석(막대 그래프 비교)
* 4. 워드클라우드 분석
* 5. LDA 토픽 모델링 분석(최적 k 탐색 포함)
* 6. word2vec 생성 후 T-sne 로 시각화
* 7. 분석 결과를 통해 얻을 수 있는 인사이트(문제점, 개선방안) 보고서 작성

# 데이터 1년치 수집

In [1]:
# !pip install selenium webdriver_manager tqdm sqlalchemy pymysql

In [23]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from datetime import datetime, timedelta
# import datetime
import pandas as pd
import time
from tqdm import tqdm
from dbio import to_db

In [27]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from datetime import datetime, timedelta
import time
import shutil


def bank_app_reviews(bank, year=1):
    options = Options()

    # ✅ headless에서는 detach 의미 거의 없음(끄는 게 안전)
    # options.add_experimental_option("detach", True)

    # ✅ DevToolsActivePort 예방용 "검증된" 옵션 조합
#     options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--remote-debugging-port=9222")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=ko-KR")

    # (선택) 크래시 감소
    options.add_argument("--disable-features=VizDisplayCompositor")

    # ✅ 잘못된 옵션 제거 (이 2줄이 문제였음)
    # options.add_argument("start-maximized")
    # options.add_argument("Chrome/135.0.0.0")
    # options.add_argument("lang=ko_KR")

    # ✅ Chrome/Chromium 바이너리 경로 자동 지정(환경마다 필요)
    chrome_bin = (shutil.which("google-chrome")
                  or shutil.which("google-chrome-stable")
                  or shutil.which("chromium")
                  or shutil.which("chromium-browser"))
    if chrome_bin:
        options.binary_location = chrome_bin

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    url = f"https://play.google.com/store/apps/details?id={bank}"
    driver.get(url)

    wait = WebDriverWait(driver, 15)

    # 평점 및 리뷰 옆의 -> 버튼 클릭
    button = wait.until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "button[aria-label*='평점 및 리뷰 자세히 알아보기']"))
    )
    button.click()

    # 최신순 정렬 버튼 클릭 (#sortBy_1은 페이지/언어/레이아웃 따라 바뀔 수 있어 예외처리 권장)
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#sortBy_1")))
    button.click()
    time.sleep(2)

    # 최신 클릭
    button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "span[aria-label*='최신']")))
    button.click()
    time.sleep(2)

    today = datetime.today()
    end_date = today - timedelta(days=year * 365)
    print(today, end_date)
    current_date = None
    pre_n_reviews = 0
    stuck = 0
    
    while True:
        # 스크롤
        driver.execute_script("document.querySelector('.fysCi.Vk3ZVd').scrollBy(0, 2000)")
        time.sleep(0.8)

        reviews = driver.find_elements(By.CSS_SELECTOR, ".RHo1pe")
        n_reviews = len(reviews)

        # 더 이상 로딩이 안 되면 종료
        # if n_reviews == pre_n_reviews:
        #     stuck += 1
        #     if stuck >= 3:
        #         print(f"\n{bank}: 더 이상 새로운 리뷰 없음. 종료.")
        #         break
        # else:
        #     stuck = 0

        # pre_n_reviews = n_reviews

        # 마지막 리뷰 날짜 파싱 시도
        try:
            date_text = reviews[-1].find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
            date_text = date_text.replace("년 ", "-").replace("월 ", "-").replace("일", "")
            current_date = datetime.strptime(date_text, "%Y-%m-%d")
        except Exception:
            # 날짜를 못 읽는 경우가 있을 수 있어 계속 진행
            pass

        print(f"{bank} 현재 리뷰 날짜: {current_date}, 리뷰수: {n_reviews} ", end="\r")

        # 목표 기간 도달하면 종료
        if current_date and current_date < end_date:
            print(f"\n{bank}: {year}년 이전 리뷰 도달. 종료.")
            break

    return driver, reviews

In [28]:
def review_extraction(driver, reviews):    
    all_result = []
    for review in tqdm(reviews):
        # 리뷰일
        review_date = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > span.bp9Aid").text
        # 리뷰일을 날짜형 데이터로 변경
        # datetime.strptime(yyyy-mm-dd, "%Y-%m-%d") 날짜형 데이터 타입으로 변환
        review_date = review_date.replace("년 ", "-").replace("월 ", "-").replace("일", "")
        review_date = datetime.strptime(review_date, "%Y-%m-%d")
        # 별점
        rating = review.find_element(By.CSS_SELECTOR, "div.Jx4nYe > div").get_attribute("aria-label").split()[3][0]
        # 사용자리뷰
        user_review = review.find_element(By.CSS_SELECTOR, "div.h3YV2d").text

        try:
            # 업체 댓글
            reply = review.find_element(By.CSS_SELECTOR, "div.ocpBU div.ras4vb > div").get_attribute("innerHTML")
        except:
            reply = None

        result = (review_date, rating, user_review, reply)
        all_result.append(result)
    result_df = pd.DataFrame(all_result, columns=['리뷰일', '평점', '사용자리뷰', '업체답변'])
#     result_df.to_csv(f"./scraping_results/{bank}_review_result.csv", index=False, encoding="utf-8-sig")
    to_db("bank_app_reviews04", bank, result_df)
    print(f"{bank} 리뷰 저장 완료")
    driver.close()  
    

In [29]:
bank_app_list = ['viva.republica.toss', 'com.kbstar.kbbank']

for bank in bank_app_list:
    driver, reviews = bank_app_reviews(bank, 1)
    review_extraction(driver, reviews)

2026-03-05 16:28:57.549640 2025-03-05 16:28:57.549640
viva.republica.toss 현재 리뷰 날짜: 2025-03-04 00:00:00, 리뷰수: 5820 
viva.republica.toss: 1년 이전 리뷰 도달. 종료.


100%|██████████████████████████████████████████████████████████████████████████████| 5820/5820 [03:44<00:00, 25.98it/s]


bank_app_reviews04 데이터베이스 확인/생성 완료
bank_app_reviews04.viva.republica.toss 데이터 저장 완료(append)
viva.republica.toss 리뷰 저장 완료
2026-03-05 17:22:06.666220 2025-03-05 17:22:06.666220
com.kbstar.kbbank 현재 리뷰 날짜: 2025-03-05 00:00:00, 리뷰수: 4180 
com.kbstar.kbbank: 1년 이전 리뷰 도달. 종료.


100%|██████████████████████████████████████████████████████████████████████████████| 4180/4180 [02:38<00:00, 26.44it/s]


bank_app_reviews04 데이터베이스 확인/생성 완료
bank_app_reviews04.com.kbstar.kbbank 데이터 저장 완료(append)
com.kbstar.kbbank 리뷰 저장 완료
